# BaseAgent - CardioKB

This notebook contains chronical records of BaseAgent's attempt in recreating CardioKB.

In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import os

from BaseAgent import BaseAgent, AgentTeam, MaxRoundsExceededError
from BaseAgent.agent_spec import AgentSpec


MCP_CONFIG = "examples/mcp_config.yaml"
SKILLS_DIR = "skills"
TEMPLATE_SRC = os.path.expanduser("~/Desktop/Cardio-KB")

In [ ]:
def _print_token_summary(agents: list):
    """Print per-agent and total token counts from accumulated usage_metrics."""
    print("\n=== Token usage ===")
    totals = {"input": 0, "output": 0, "total": 0}
    for agent in agents:
        metrics = agent.usage_metrics
        input_tokens = sum(m.input_tokens or 0 for m in metrics)
        output_tokens = sum(m.output_tokens or 0 for m in metrics)
        total_tokens = sum(m.total_tokens or 0 for m in metrics)
        cost = sum(m.cost or 0.0 for m in metrics)
        cost_str = f"  ${cost:.4f}" if cost else ""
        print(f"  {agent.spec.name}: {input_tokens} in / {output_tokens} out / {total_tokens} total{cost_str}")
        totals["input"] += input_tokens
        totals["output"] += output_tokens
        totals["total"] += total_tokens
    print(f"  {'─' * 40}")
    print(f"  all agents:  {totals['input']} in / {totals['output']} out / {totals['total']} total")

# Run a KG-building agentic AI team

In [ ]:
# ----- Ontology Agent -----
ontology_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="ontology_agent",
        role=(
            "A biomedical ontology engineer managing the OWL schema and project configuration. "
            "You own ontology/cardiokb_ontology.rdf and src/ontology_configs.py: update disease_scope "
            "(primary_terms, umls_cuis, doid_ids, mesh_ids) and keep node_types/edge_types in "
            "sync with the RDF. Only modify the RDF on explicit request. Never edit Python source files."
        ),
        llm="claude-haiku-4-5",
        skill_names=["ontology-protocol"],
    ),
    require_approval="never",
)
ontology_agent.add_mcp(MCP_CONFIG)

# ----- Engineer Agent -----
engineer_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="engineer_agent",
        role=(
            "A Python software engineer writing parsers under src/parsers/. "
            "Each parser inherits from BaseParser and downloads data from one biomedical source, "
            "returning clean pandas DataFrames. Follow the full registration checklist: "
            "src/parsers/__init__.py, src/main.py PARSERS dict, test/eval_parser.py PARSER_CLASS_MAP. "
            "Run `python src/main.py --source <name>` to verify each parser produces TSVs. "
            "Never hardcode any disease-specific values in the parser code. "
            "Never hardcode any credentials."
            "You do not modify OWL files or src/ontology_configs.py."
        ),
        llm="claude-sonnet-4-6",
        skill_names=["parser-protocol"],
    ),
    require_approval="never",
)
engineer_agent.add_mcp(MCP_CONFIG)

# ----- Mapping Agent -----
mapping_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="mapping_agent",
        role=(
            "A knowledge graph mapping specialist owning src/ontology_configs.py. "
            "You map processed TSV columns to OWL node types and relationship types. "
            "Other useful information such as gene descriptions, annotations, structures, and cross-references is included as properties. "
            "Always place node entries before relationship entries. "
            "Verify all OWL names against src/ontology_configs.py node_types/edge_types before writing. "
            "Never edit Python source files."
        ),
        llm="claude-haiku-4-5",
        skill_names=["mapping-protocol"],
    ),
    require_approval="never",
)
mapping_agent.add_mcp(MCP_CONFIG)

# ----- Memgraph Agent -----
memgraph_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="memgraph_agent",
        role=(
            "A graph database engineer who runs the full pipeline and validates graph export. "
            "Run `python src/main.py` inside the repo to produce data/output/cardiokb_populated.rdf, "
        ),
        llm="claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
memgraph_agent.add_mcp(MCP_CONFIG)

# ----- Evaluator Agent -----
evaluator_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="evaluator_agent",
        role=(
            "A KG quality evaluator who runs eval_after_parser.py, eval_after_ontology.py, "
            "and eval_after_memgraph.py in sequence. Report tier-1 blocking failures "
            "(zero node/edge counts, OWL conformance < 1.0) and overall KG quality. "
            "Flag any blocking failures that must be resolved before the KG is used."
        ),
        llm="claude-haiku-4-5",
        skill_names=["evaluation-protocol"],
    ),
    require_approval="never",
)
evaluator_agent.add_mcp(MCP_CONFIG)

# ----- HITL Agent -----
hitl_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="hitl_agent",
        role=(
            "A human review coordinator. Summarize the previous agent's output in less than 5 bullet "
            "points with descriptions on the task, target file(s), and rationale. "
            "Then call 'ask_user' with the summary and a clear yes/no question. "
        ),
        llm="claude-haiku-4-5",
        skill_names=["hitl-protocol"],
    ),
    require_approval="never",
)
hitl_agent.add_mcp(MCP_CONFIG)

In [ ]:

agents = [
    ontology_agent, engineer_agent, evaluator_agent, mapping_agent, hitl_agent,
]

team = AgentTeam(
    agents=agents,
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=100,
)


TARGET_DATABASE = "ClinVar"
THREAD_ID = TARGET_DATABASE

task = f"""
Coordinate a team of agents to update and evaluate the {TARGET_DATABASE} parser in the repo at {TEMPLATE_SRC}. 

Available agents and their responsibilities:
- ontology_agent: Manage OWL schema in ontology/cardiokb_ontology.rdf and src/ontology_configs.py — ensure the project strictly conforms to the ontology.
- engineer_agent: Manage parsers in src/parsers/. Verify with `python src/main.py --source <name>` run inside the repo.
- mapping_agent: Manage src/ontology_configs.py — ensure the correct configurations (RDF classes, data property map, and merge/cross-reference).
- evaluator_agent: Focus on parser-specific evaluation by running only eval_after_parser script and report the quality summary. List any tier-1 blocking failures explicitly.
- hitl_agent: Pause for user review after collecting all proposed changes to config files (src/ontology_configs.py). Relay any feedback back to the responsible agent.


Instructions:
- Find more information about the {TARGET_DATABASE} data and the download file contents from the official {TARGET_DATABASE} documentation.
- Include useful information as properties. For drugs, useful information includes the drug description, index, structure (in SMILES and InChI formats), FDA approval status, and more. For diseases, useful information include disease descriptions, symptoms, and other relevant details. For genes, useful information includes expression, call quality, FDR, expression score, and expression rank.
- Coordinate across agents to produce a verified {TARGET_DATABASE} parser with no tier-1 blocking failures.
- Add new RDF classes when and only when the extracted data can be cross referenced to other sources using standardized identifiers. Free-text descriptions can only be included as properties, not RDF classes.
- Upon user approval, update OWL schema in ontology/cardiokb_ontology.rdf and graph schema in src/ontology_configs.py to accommodate the new RDF classes.
- Upon user approval, update "data_property_map" in src/ontology_configs.py to accommodate the new RDF properties.


Inter-agent contracts:
- All parsers must be verified before evaluator_agent runs.
- mapping_agent must report valid cross-reference information before the engineer_agent starts.
- On tier-1 failures, route back to engineer_agent to fix before re-evaluating.


Strict contracts and constraints:
- Do not include any filter (e.g., disease, tissue, or drug) if the specific parser doesn't include these RDF classes.
- Directly retrieve data from the source database. Do not use any precomputed third-party processed data. Do not use any precomputed data from Hetionet.
- src/ontology_configs.py sets project.name, display_name, and all disease_scope fields (primary_terms, umls_cuis, doid_ids, mesh_ids). 
- The PARSERS key, PARSER_CLASS_MAP key, ontology_configs.py prefix, and data/processed/ subdirectory name must all be identical strings.
- In src/ontology_configs.py, all node entries must precede all relationship entries.
- OWL names in src/ontology_configs.py must be active (uncommented) in the ontology.
- Credentials use the _env suffix in parser args; the parser constructor must accept the stripped parameter name.
"""

In [ ]:
_log, result = team.run_sync(task, thread_id=THREAD_ID)

In [ ]:
_log, result = team.run_sync("Continue", thread_id=THREAD_ID)

# Sub-agent follow-up

In [ ]:
team_tid = team._current_thread_id
_log2, result2 = evaluator_agent.run(
    f"Report the evaluation status of the {TARGET_DATABASE} parser",
    thread_id=f"{team_tid}:evaluator_agent"
)

# Token usage summary

In [ ]:
_print_token_summary(agents)